In [1]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 15.8 MB/s eta 0:00:00


In [2]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

print("Model loaded.")

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Model loaded.


In [4]:
from qwen_vl_utils import process_vision_info

def call_vlm(prompt: str, images: list = None) -> str:
    """
    prompt: text instruction/question
    images: list of local image file paths (optional)
    Returns: model's text response as a string
    """
    content = []
    if images:
        for img_path in images:
            content.append({"type": "image", "image": img_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_special_tokens=False
    )
    return output_text[0]

In [6]:
# Text-only test
print(call_vlm("What is 2+2?"))

# With an image (upload one via the folder icon on the left sidebar first)
# print(call_vlm("What's in this image?", images=["/content/your_test_image.jpg"]))

2+2 is equal to 4.


In [8]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [9]:
from PIL import Image

img_path = "/content/Gemini_Generated_Image_3cq4s23cq4s23cq4.png"
img = Image.open(img_path)
img = img.resize((512, 512))
img.save("/content/resized_test.png")

In [14]:
print(call_vlm("What's in this image?", images=["/content/resized_test.png"]))

The image is a collage that showcases two different applications and their respective use cases. 

1. **The Solo Burden**:
   - This application appears to be designed for caregivers, particularly mothers, who are managing the care of a baby and handling household chores.
   - The interface includes reminders for medication and blood pressure checks, indicating its focus on health and wellness management.
   - The application seems to be designed to help caregivers manage their tasks efficiently and stay organized.

2. **The Distributed Future with Amara**:
   - This application is presented as a live working prototype, suggesting it is a new or experimental app.
   - It shows a family setting where multiple family members are interacting with the app, indicating its use in a household context.
   - The app appears to be designed for managing tasks and coordinating activities among family members, promoting a distributed approach to household management.

The image highlights the poten

In [12]:
from PIL import Image
from qwen_vl_utils import process_vision_info

def call_vlm(prompt: str, images: list = None) -> str:
    content = []
    if images:
        for img_path in images:
            img = Image.open(img_path)
            img.thumbnail((512, 512))  # resize in-place, preserves aspect ratio, prevents OOM
            resized_path = img_path.rsplit(".", 1)[0] + "_resized.png"
            img.save(resized_path)
            content.append({"type": "image", "image": resized_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_special_tokens=False)
    return output_text[0]

In [3]:
%%writefile call_vlm.py
import torch
from PIL import Image
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

model_id = "Qwen/Qwen2-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_id, quantization_config=bnb_config, device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

def call_vlm(prompt: str, images: list = None) -> str:
    content = []
    if images:
        for img_path in images:
            img = Image.open(img_path)
            img.thumbnail((512, 512))
            resized_path = img_path.rsplit(".", 1)[0] + "_resized.png"
            img.save(resized_path)
            content.append({"type": "image", "image": resized_path})
    content.append({"type": "text", "text": prompt})

    messages = [{"role": "user", "content": content}]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=512)
    generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_special_tokens=False)
    return output_text[0]

Writing call_vlm.py


In [4]:
%%writefile generator.py
from call_vlm import call_vlm


def _build_prompt(query, text_chunks, images):
    lines = [
        "Answer the question using only the context below. "
        "If the context does not contain the answer, say you don't know.",
        "",
        "Context:",
    ]
    for i, chunk in enumerate(text_chunks, 1):
        lines.append(f"[Text {i} | source: {chunk['source_doc']}]\n{chunk['text']}")

    for i, img in enumerate(images, 1):
        caption = img.get("caption") or "no caption"
        lines.append(f"[Image {i} | source: {img['source_doc']}] {caption}")

    lines.append("")
    lines.append(f"Question: {query}")
    lines.append("Answer:")
    return "\n\n".join(lines)


def generate_answer(query, retrieved):
    text_chunks = retrieved.get("text_chunks") or []
    images = retrieved.get("images") or []

    prompt = _build_prompt(query, text_chunks, images)
    image_paths = [img["path"] for img in images]

    answer = call_vlm(prompt, images=image_paths or None)

    return {
        "answer": answer,
        "used_text": text_chunks,
        "used_images": images,
    }

Writing generator.py


In [5]:
from generator import generate_answer
from PIL import Image

# create a stub test image
img = Image.new("RGB", (200, 200), color="blue")
img.save("/content/stub1.png")

retrieved = {
    "text_chunks": [
        {"text": "The Eiffel Tower is located in Paris, France.", "source_doc": "wiki_paris.txt"}
    ],
    "images": [
        {"path": "/content/stub1.png", "caption": "A blue placeholder image", "source_doc": "stub_source"}
    ],
}

result = generate_answer("Where is the Eiffel Tower?", retrieved)
print(result["answer"])

config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/56.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/730 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

The Eiffel Tower is located in Paris, France.


In [2]:
!pip install -q qwen-vl-utils

In [1]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 18.5 MB/s eta 0:00:00


In [6]:
%%writefile test_generator.py
import os

from PIL import Image, ImageDraw

from generator import generate_answer

STUB_IMAGE_DIR = "test_assets"


def make_stub_image(path, text, color):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    img = Image.new("RGB", (400, 300), color=color)
    draw = ImageDraw.Draw(img)
    draw.text((20, 20), text, fill="black")
    img.save(path)


def print_result(label, result):
    print(f"=== {label}: ANSWER ===")
    print(result["answer"])
    print("\n=== USED TEXT ===")
    for chunk in result["used_text"]:
        print(f"- ({chunk['source_doc']}) {chunk['text'][:80]}...")
    print("\n=== USED IMAGES ===")
    for img in result["used_images"]:
        print(f"- ({img['source_doc']}) {img['path']} - {img['caption']}")
    print()


def test_text_and_images():
    img1_path = os.path.join(STUB_IMAGE_DIR, "chart.png")
    img2_path = os.path.join(STUB_IMAGE_DIR, "diagram.png")
    make_stub_image(img1_path, "Revenue chart 2023", "lightblue")
    make_stub_image(img2_path, "System diagram", "lightgreen")

    retrieved = {
        "text_chunks": [
            {
                "text": "The company reported revenue of $5.2 million in 2023, "
                        "up 18% year over year.",
                "source_doc": "annual_report_2023.pdf",
            },
            {
                "text": "Growth was driven primarily by the new enterprise product line.",
                "source_doc": "annual_report_2023.pdf",
            },
        ],
        "images": [
            {
                "path": img1_path,
                "caption": "Bar chart showing revenue by quarter in 2023",
                "source_doc": "annual_report_2023.pdf",
            },
            {
                "path": img2_path,
                "caption": "Diagram of the enterprise product architecture",
                "source_doc": "product_overview.pdf",
            },
        ],
    }

    query = "What was the company's revenue in 2023 and what drove its growth?"
    result = generate_answer(query, retrieved)
    print_result("TEXT + IMAGES", result)


def test_text_only():
    retrieved = {
        "text_chunks": [
            {
                "text": "The company reported revenue of $5.2 million in 2023, "
                        "up 18% year over year.",
                "source_doc": "annual_report_2023.pdf",
            },
            {
                "text": "Growth was driven primarily by the new enterprise product line.",
                "source_doc": "annual_report_2023.pdf",
            },
        ],
        "images": [],
    }

    query = "What was the company's revenue in 2023 and what drove its growth?"
    result = generate_answer(query, retrieved)
    print_result("TEXT ONLY", result)


def test_images_only():
    img1_path = os.path.join(STUB_IMAGE_DIR, "chart.png")
    img2_path = os.path.join(STUB_IMAGE_DIR, "diagram.png")
    make_stub_image(img1_path, "Revenue chart 2023", "lightblue")
    make_stub_image(img2_path, "System diagram", "lightgreen")

    retrieved = {
        "text_chunks": [],
        "images": [
            {
                "path": img1_path,
                "caption": "Bar chart showing revenue by quarter in 2023",
                "source_doc": "annual_report_2023.pdf",
            },
            {
                "path": img2_path,
                "caption": "Diagram of the enterprise product architecture",
                "source_doc": "product_overview.pdf",
            },
        ],
    }

    query = "What was the company's revenue in 2023 and what drove its growth?"
    result = generate_answer(query, retrieved)
    print_result("IMAGES ONLY", result)


def main():
    test_text_and_images()
    test_text_only()
    test_images_only()


if __name__ == "__main__":
    main()


Writing test_generator.py


In [7]:
!python test_generator.py

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files: 100% 5/5 [00:00<00:00, 1161.21it/s]
Download complete: :           |  0.00B            
Download complete: :           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B            
Loading weights: 100% 730/730 [01:07<00:00, 10.85it/s]
=== TEXT + IMAGES: ANSWER ===
The company's revenue in 2023 was $5.2 million, which was up 18% year over year. Growth was driven primarily by the new enterprise product line.

=== USED TEXT ===
- (annual_report_2023.pdf) The company reported revenue of $5.2 million in 2023, up 18% year over year....
- (annual_report_2023.pdf) Growth was driven primarily by the new enterprise product line....

=== USED IMAGES ===
- (annual_report_2023.pdf) test_assets/chart.png - Bar chart showing revenue by quarter in 2023
- (product_overview.pdf) test_assets/diagram.png - Diagram of the enterprise product architecture

=== TEXT ONLY: ANSWER ===
The 